In [ ]:
#Bài tậ 1 về nhà giải thuật AKT vào bài toán 15 puzzle (n > 4)
import heapq

class Node:
  def __init__ (self, state, parent, action, g, h, zero_idx):
    self.state = state
    self.parent = parent
    self.action = action
    self.g = g
    self.h = h
    self.f = g + h
    self.zero_idx = zero_idx

  def __lt__ (self, other):
    return self.f < other.f

def get_target_positions(target_state, n):
  return {val: (i // n, i % n) for i, val in enumerate(target_state) if val != 0}

def calculate_manhattan(state, target_positions, n):
  distance = 0
  for i, val in enumerate(state):
    if val != 0:
      cur_r = i//n
      cur_c = i%n
      target_r, target_c = target_positions[val]
      distance += abs(cur_r - target_r) + abs(cur_c - target_c)
  return distance

def print_matrix(state, n):
  for i in range(0, n*n, n):
    row = state[i:i+n]
    print("".join(f"{val:2}" for val in row))
  print()

def solve_n_puzzle(initial_matrix):
  n = len(initial_matrix)
  target_list = list(range(1, n*n)) + [0]
  target_state = tuple(target_list)

  initial_state = tuple(val for row in initial_matrix for val in row)
  target_positions = get_target_positions(target_state, n)
  moves_config = {
      'LÊN': -n, 'XUỐNG': n, 'TRÁI': -1, 'PHẢI': 1
  }
  pq = []
  zero_idx = initial_state.index(0)
  h_initial = calculate_manhattan(initial_state, target_positions, n)
  heapq.heappush(pq, Node(initial_state, None, None, 0, h_initial, zero_idx))

  visited = {initial_state: 0} # Lưu trữ g_score để tối ưu
  nodes_explored = 0

  while pq:
      current = heapq.heappop(pq)
      nodes_explored += 1

      if current.state == target_state:
          print(f"== Đã giải xong (N={n}) ==")
          print(f"Số trạng thái đã duyệt: {nodes_explored}")
          print(f"Số bước đi: {current.g}\n")
          return

      curr_r, curr_c = current.zero_idx // n, current.zero_idx % n

      for move_name, move_val in moves_config.items():

          if move_name == 'LÊN' and curr_r == 0: continue
          if move_name == 'XUỐNG' and curr_r == n - 1: continue
          if move_name == 'TRÁI' and curr_c == 0: continue
          if move_name == 'PHẢI' and curr_c == n - 1: continue

          new_zero_idx = current.zero_idx + move_val

          state_list = list(current.state)
          state_list[current.zero_idx], state_list[new_zero_idx] = \
              state_list[new_zero_idx], state_list[current.zero_idx]
          new_state = tuple(state_list)


          new_g = current.g + 1
          if new_state not in visited or new_g < visited[new_state]:
              visited[new_state] = new_g
              h_new = calculate_manhattan(new_state, target_positions, n)
              heapq.heappush(pq, Node(new_state, current, move_name, new_g, h_new, new_zero_idx))

  print("Không tìm thấy lời giải.")

if __name__ == "__main__":
    initial_5x5 = [
        [1, 2, 3, 4, 5],
        [6, 7, 8, 9, 10],
        [11, 12, 13, 14, 15],
        [16, 17, 18, 19, 20],
        [21, 22, 0, 23, 24]
    ]
    solve_n_puzzle(initial_5x5)

== Đã giải xong (N=5) ==
Số trạng thái đã duyệt: 3
Số bước đi: 2



In [9]:
import heapq

def print_matrix(matrix):
    print("\nMa trận khoảng cách (TP)")
    n = len(matrix)
    for row in matrix:
        print(" ".join(f"{val:4}" for val in row))
    print("-" * 20)

def get_heuristic(current_city, unvisited, matrix):
    if not unvisited:
        return matrix[current_city][0]
    distances = [matrix[current_city][city] for city in unvisited]
    return min(distances)

def solve_tsp(matrix):
    n = len(matrix)
    start_city = 0
    start_h = get_heuristic(start_city, set(range(1, n)), matrix)
    pq = [(start_h, 0, start_city, (start_city,), [start_city])]
    visited_states = {}

    while pq:
        f, g, u, visited, path = heapq.heappop(pq)
        if len(visited) == n:
            total_g = g + matrix[u][start_city]
            return path + [start_city], total_g
        for v in range(n):
            if v not in visited:
                new_g = g + matrix[u][v]
                new_visited = tuple(sorted(list(visited) + [v]))
                state = (v, new_visited)
                if state not in visited_states or new_g < visited_states[state]:
                    visited_states[state] = new_g
                    unvisited = set(range(n)) - set(new_visited)
                    h = get_heuristic(v, unvisited, matrix)
                    heapq.heappush(pq, (new_g + h, new_g, v, new_visited, path + [v]))
    return None, 0
if __name__ == "__main__":
    dist_matrix = [
        [0, 20, 10, 15],
        [20, 0, 35, 25],
        [10, 35, 0, 30],
        [15, 25, 30, 0]
    ]

    print_matrix(dist_matrix)
    best_path, total_distance = solve_tsp(dist_matrix)
    if best_path:
        print("KẾT QUẢ:")
        print(f"Lộ trình: {' -> '.join(map(str, best_path))}")
        print(f"Tổng quãng đường: {total_distance}")


Ma trận khoảng cách (TP)
   0   20   10   15
  20    0   35   25
  10   35    0   30
  15   25   30    0
--------------------
KẾT QUẢ:
Lộ trình: 0 -> 2 -> 3 -> 1 -> 0
Tổng quãng đường: 85
